In [ ]:
import os
import re
import warnings
import rasterio
import numpy as np
from glob import glob
from tqdm.auto import tqdm
from rasterio.windows import Window
from typing import List, Tuple

In [ ]:
def apply_binary_mask(
    data_raster_path: str,
    mask_raster_path: str,
    output_path: str,
    ) -> None:
    """
    Apply a binary mask to a data raster in a block-wise fashion.

    Args:
        data_raster_path (str): Path to the input raster whose values are being
            masked. May have any number of bands. Its dtype must be able to hold
            the fill value of -1, so signed or floating-point types are required
        mask_raster_path (str): Path to the binary mask raster. Only band 1 is
            read. Must be pixel-aligned with the data raster, i.e. identical CRS, affine transform, width, and height. Pixels equal to 0 are masked.
        output_path (str): Path to write the masked raster. Written with the
            same driver and profile as the data raster, so the extension should match that format. 

    """
    block_w, block_h = 4096, 4096

    with rasterio.open(data_raster_path) as src_data, rasterio.open(mask_raster_path) as src_mask:
        # Alignment checks
        if (src_data.crs != src_mask.crs or
            src_data.transform != src_mask.transform or
            src_data.width != src_mask.width or
            src_data.height != src_mask.height):
            raise ValueError("Data and mask rasters must be aligned (same CRS, transform, width, height).")

        # Prepare output profile (preserve metadata as-is)
        profile = src_data.profile.copy()

        with rasterio.open(output_path, "w", **profile) as dst:
            
            # Copy dataset-level tags
            dst.update_tags(**src_data.tags())

            # Copy per-band descriptions and tags
            for b in range(1, src_data.count + 1):
                desc = src_data.descriptions[b - 1]
                if desc:
                    dst.set_band_description(b, desc)
                band_tags = src_data.tags(b)
                if band_tags:
                    dst.update_tags(b, **band_tags)

            width, height = src_data.width, src_data.height
            n_cols = (width  + block_w - 1) // block_w
            n_rows = (height + block_h - 1) // block_h
            total_windows = n_cols * n_rows

            # Iterate manual 4096x4096 windows with progress bar
            pbar = tqdm(total=total_windows, desc="Masking blocks (4096x4096)", mininterval=1)
            try:
                for row_off in range(0, height, block_h):
                    rows = min(block_h, height - row_off)
                    for col_off in range(0, width, block_w):
                        cols = min(block_w, width - col_off)
                        window = Window(col_off, row_off, cols, rows)

                        # Read data (all bands) and mask (first band)
                        data_block = src_data.read(window=window)        # (bands, rows, cols)
                        mask_block = src_mask.read(1, window=window)     # (rows, cols)

                        # Set the MTBS raster to -1 where out where mask == 0 (apply to all bands)
                        zero_where = (mask_block == 0)[None, :, :]       # broadcast to (1, rows, cols)
                        out_block = data_block.copy()
                        out_block[zero_where] = -1

                        dst.write(out_block, window=window)
                        pbar.update(1)
            finally:
                pbar.close()

    return None


In [ ]:
# Define a path to the project folder
project_folder = "<PATH/TO/PROJECT/FOLDER>"

# Define a path to the folder that contains all of the raster datasets
mtbs_raster_folder = f"{project_folder}/data/rasters/mtbs_severity_annual"

# Define a path to all-years NLCD mask
nlcd_raster = f"{project_folder}/data/rasters/ncld_forest_mask/nlcd_forest_mask_1984_2024_mosaic.tif"

# Define a path to the output folder
output_folder = f"{project_folder}/data/rasters/mtbs_severity_annual_masked"
if not os.path.exists(output_folder):
    os.mkdir(output_folder)

In [ ]:
# Aggregate all of the raster files that need to be processed
all_mtbs_raster_paths = glob(f"{mtbs_raster_folder}/*.tif")

# Aggregate all of the nlcd masks
all_nlcd_raster_paths = [nlcd_raster] * len(all_mtbs_raster_paths)

In [ ]:
# Pair up the MTBS and the NLCD rasters
pairs = list(zip(all_mtbs_raster_paths, all_nlcd_raster_paths))

In [ ]:
# Iterate over the rasters to process
for mtbs_raster_path, nlcd_raster_path in tqdm(pairs, desc="Processing rasters..."):

    # Format the output path
    output_raster_path = f"{output_folder}/{os.path.basename(mtbs_raster_path)}"

    # Apply the mask
    apply_binary_mask(
        data_raster_path = mtbs_raster_path,
        mask_raster_path = nlcd_raster_path,
        output_path = output_raster_path
        )